In [15]:
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os 
import seaborn as sns
import cv2
import random
import datetime

In [16]:
from keras.models import Sequential, Model, load_model
from keras.layers import Dense,Dropout,Flatten,Conv2D,MaxPooling2D,Input,Activation,GlobalAveragePooling2D, BatchNormalization,Reshape
from keras.optimizers import Adam, RMSprop
from keras.layers import LeakyReLU,Conv2DTranspose
from keras.preprocessing.image import ImageDataGenerator
from keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
# from keras.utils import plot_model
from keras.datasets.cifar10 import load_data

In [17]:
def define_discriminator(in_shape=(32,32,3)):
    model = Sequential()
    
    model.add(Conv2D(64, (3,3), padding='same', input_shape=in_shape))
    model.add(LeakyReLU(alpha=0.2))
    
    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(128, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(256, (3,3), strides=(2,2), padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # classifier
    model.add(Flatten())
    model.add(Dropout(0.4))
    model.add(Dense(1,activation='sigmoid'))
    opt = Adam(lr=0.0002, beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt,metrics=['accuracy'])

    return model

def define_generator(latent_dim):
    model = Sequential()

    n_nodes = 256 * 4 * 4
    model.add(Dense(n_nodes,input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(Reshape((4,4,256)))
    # upsample to 8x8
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # upsample to 16x16
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))
    # 
    model.add(Conv2DTranspose(128,(4,4),strides=(2,2),padding='same'))
    model.add(LeakyReLU(alpha=0.2))

    model.add(Conv2D(3,(3,3),activation='tanh',padding='same'))
    return model

def define_gan(g_model,d_model):
    d_model.trainable = False
    model = Sequential()
    model.add(g_model)
    model.add(d_model)
    opt = Adam(lr=0.0002,beta_1=0.5)
    model.compile(loss='binary_crossentropy',optimizer=opt)
    return model

In [18]:
def load_real_samples():
    # load cifar10 dataset
    (trainX,_) , (_,_) = load_data()
    X = trainX.astype('float32')
    # normalize to [-1,1]
    X = (X - 127.5) / 127.5
    return X

def generate_real_samples(dataset,n_samples):
    # choose random instances
    ix = np.random.randint(0,dataset.shape[0],n_samples)
    # retrieve selected images
    X = dataset[ix]
    # generate 'real' class labels (1)
    y = np.ones((n_samples,1))
    return X,y


def generate_latent_points(latent_dim,n_samples):
    # generate points in the latent space
    x_input = np.random.randn(latent_dim*n_samples)
    # reshape
    x_input = x_input.reshape(n_samples,latent_dim)
    
    return x_input

def generate_fake_samples(g_model,latent_dim,n_samples):
    # generate points in latent space
    x_input = generate_latent_points(latent_dim,n_samples)
    # predict outputs
    X = g_model.predict(x_input)
    # create 'fake' class labels (0)
    y = np.zeros((n_samples,1))
    return X,y


def save_plot(examples,epoch,n=10):
    # plot images
    for i in range(n*n):
        plt.subplot(n,n,1+i)
        plt.axis('off')
        plt.imshow(examples[i,:,:,0],cmap='gray_r')
    # save plot to file
    filename = 'generated_plot_e%03d.png' % (epoch+1)
    plt.savefig(filename)
    plt.close()


def summarize_performance(epoch,g_model,d_model,dataset,latent_dim,n_samples=100):
    X_real,y_real = generate_real_samples(dataset,n_samples)
    _,acc_real = d_model.evaluate(X_real,y_real,verbose=0)
    X_fake,y_fake = generate_fake_samples(g_model,latent_dim,n_samples)
    _,acc_fake = d_model.evaluate(X_fake,y_fake,verbose=0)
    print('Accuracy real: %.0f%%, fake: %.0f%%' % (acc_real*100,acc_fake*100))
    save_plot(X_fake,epoch)
    filename = 'generator_model_%03d.h5' % (epoch+1)
    g_model.save(filename)



def train(g_model,d_model,gan_model,dataset,latent_dim,n_epochs=200,n_batch=128):
    bat_per_epo = int(dataset.shape[0]/n_batch)
    half_batch = int(n_batch/2)
    for i in range(n_epochs):
        for j in range(bat_per_epo):
            # get randomly selected 'real' samples
            X_real,y_real = generate_real_samples(dataset,half_batch)
            # update discriminator model weights on real samples
            d_loss1,_ = d_model.train_on_batch(X_real,y_real)
            # generate 'fake' examples
            X_fake,y_fake = generate_fake_samples(g_model,latent_dim,half_batch)
            # update discriminator model weights on fake samples
            d_loss2,_ = d_model.train_on_batch(X_fake,y_fake)
            # prepare points in latent space as input for the generator
            X_gan = generate_latent_points(latent_dim,n_batch)
            # create inverted labels for the fake samples
            y_gan = np.ones((n_batch,1))
            # update the generator via the discriminator's error
            g_loss = gan_model.train_on_batch(X_gan,y_gan)
            # summarize loss on this batch
            print('>%d, %d/%d, d1=%.3f, d2=%.3f g=%.3f' % (i+1,j+1,bat_per_epo,d_loss1,d_loss2,g_loss))

        if (i+1) % 10 == 0:
            summarize_performance(i,g_model,d_model,dataset,latent_dim)



In [19]:
latent_dim = 100

d_model = define_discriminator()
g_model = define_generator(latent_dim)
gan_model = define_gan(g_model,d_model)
dataset = load_real_samples()
train(g_model,d_model,gan_model,dataset,latent_dim)

2/2 [==============================] - 0s 6ms/step
>1, 1/390, d1=0.704, d2=0.696 g=0.691
2/2 [==============================] - 0s 5ms/step
>1, 2/390, d1=0.631, d2=0.698 g=0.689
2/2 [==============================] - 0s 4ms/step
>1, 3/390, d1=0.556, d2=0.705 g=0.682
2/2 [==============================] - 0s 5ms/step
>1, 4/390, d1=0.464, d2=0.725 g=0.664
2/2 [==============================] - 0s 4ms/step
>1, 5/390, d1=0.368, d2=0.773 g=0.628
2/2 [==============================] - 0s 5ms/step
>1, 6/390, d1=0.255, d2=0.861 g=0.586
2/2 [==============================] - 0s 4ms/step
>1, 7/390, d1=0.177, d2=0.941 g=0.573
2/2 [==============================] - 0s 7ms/step
>1, 8/390, d1=0.174, d2=0.923 g=0.610
2/2 [==============================] - 0s 5ms/step
>1, 9/390, d1=0.191, d2=0.820 g=0.706
2/2 [==============================] - 0s 5ms/step
>1, 10/390, d1=0.173, d2=0.673 g=0.885
2/2 [==============================] - 0s 6ms/step
>1, 11/390, d1=0.217, d2=0.550 g=1.024
2/2 [==============

KeyboardInterrupt: 